# 01 · Basics — the one chain

`rxembed` embeds a conformer ensemble for anything from a bare SMILES to a frozen bimetallic TS, then searches, dedups, and scores it — **the same chain every time**, each capability just *one more argument*.

This notebook: the `embed → mc → prune` chain and what it returns, the **mutation contract**, all four **dedup** methods, the **geometry gate** (proof the periphery is chemically clean), and the ensemble **landscape**. Every later notebook is this chain with more arguments.

In [ ]:
import tempfile

import rxembed as rx
import xyzrender
from rxembed import dedup
import matplotlib.pyplot as plt

rx.set_verbose("INFO")  # one log line per stage; 'DEBUG' adds constraint values
ACID = "O=C(O)CCCCc1ccccc1"  # 5-phenylpentanoic acid — a flexible running example

## The chain

`embed()` returns an **`Ensemble`**: one RDKit `Mol` (the graph) holding many **conformers** (geometries). `.mc()` runs an openconf torsional search; `.prune()` dedups. They **build in place and chain**.

In [ ]:
ens = rx.embed(ACID).mc().prune()
print("one Mol, many conformers:", type(ens.mol).__name__, "|", len(ens), "conformers")
print(repr(ens))

## The mutation contract

Steps that **build** the ensemble (`mc`, `minimize`, `prune`) change it in place and return it, so they chain. Steps that **derive** a smaller set (`representatives`, `lowest`, `align`, and indexing `ens[i]` / `ens[:k]`) return a **new** Ensemble and leave the parent untouched. Looking never mutates.

In [ ]:
base = rx.embed(ACID).mc().minimize()  # minimize gives FF energies, so lowest() can rank
reps = base.representatives()  # DERIVES a new ensemble — parent untouched
best3 = base.lowest(3)  # DERIVES the 3 lowest-energy conformers
print(f"derive: parent still {len(base)} confs; representatives()->{len(reps)}, lowest(3)->{len(best3)}")
print(f"index:  reps[1] -> {reps[1].n}-conformer Ensemble | base[:5] -> {base[:5].n} confs (also a derive)")
n_before = len(base)
base.prune()  # BUILDS in place — parent changes
print(f"build:  prune() {n_before} -> {len(base)} confs (mutated in place)")

## Dedup — four ways

`prune(by=...)` dedups geometrically. **`rmsd`** (the safe default) is rigorous everywhere; **`moi`** is a fast coarse pre-filter; **`descriptor`** compares torsion fingerprints; **`energy`** collapses near-degenerate minima (prism-free). Nothing is lost — dropped frames stay in `ens.mol`.

In [ ]:
p = rx.embed(ACID).mc().minimize()
E = [p.energies[i] for i in p.ids]
for m in ("rmsd", "moi", "descriptor", "energy"):
    kept, _ = dedup.apply(p.mol, p.ids, E, method=m)
    print(f"  {m:11s}: {len(p.ids):3d} -> {len(kept):3d} kept")

## The geometry gate

`rx.geometry.check` is the **bad-geometry validator**: broken conjugation (twisted amide), bad H positions, non-bonded clashes, stretched bonds, moved frozen core. It is the universal acceptance test — every notebook calls it. A clean minimized molecule passes with zero violations.

In [ ]:
rep = rx.geometry.check(p.mol, conf_id=p.ids[0])  # 0 violations: the FF-relaxed periphery is chemically sound
print("geometry.check:", "CLEAN ✓" if rep.ok() else f"FAIL ✗ {rep.violations[:6]}")

## The landscape

`landscape()` projects the ensemble to 2D (PCA / t-SNE) — see the conformer diversity and, later, which points a prune kept vs dropped. `representatives()` returns one lowest-energy conformer per mode.

In [ ]:
# The ensemble map: kept modes coloured, and the conformers a prune MERGED AWAY drawn as faint small
# points in the same projection — so you see the whole ensemble and which regions collapsed to a rep.
ens = rx.embed(ACID).mc().prune()  # prune() populates ens.discarded
print(f"{len(ens)} kept, {len(ens.discarded)} merged away; active latent: {dedup.active_blocks(ens.mol, ens.ids)}")
ens.landscape("pca", color="cluster")  # modes 1..n coloured; 'dropped (N)' faint
plt.title("conformer landscape — kept modes + merged (faint)")
plt.show()
xyzrender.render(
    xyzrender.load(
        ens.representatives().align().dump(tempfile.mktemp(suffix=".xyz")), ensemble=True, ensemble_color="spectral"
    ),
    no_hy=True,
)  # one representative structure per mode

## Dump at any stage

`.dump()` writes the current conformers as a multi-frame `.xyz` — at any point in the chain, or from any derived ensemble. This is how you hand geometries to another tool.

In [ ]:
import os  # tempfile already imported above

d = tempfile.mkdtemp()
e = rx.embed(ACID, seed=1)
e.dump(f"{d}/seeds.xyz")  # raw embedded seeds
e.mc().prune().representatives().dump(f"{d}/reps.xyz")  # one frame per mode
print("dumped multi-frame xyz:", sorted(os.listdir(d)))